In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from pymongo import MongoClient
import time

# --- CONFIGURACIÓN INICIAL ---
NOMBRE_GRUPO = "DataTraders"

# Diccionario con las URLs de Yahoo Finance
URLS_OBJETIVO = {
    "gainers": "https://finance.yahoo.com/markets/stocks/gainers/",
    "losers": "https://finance.yahoo.com/markets/stocks/losers/",
    "most_active": "https://finance.yahoo.com/markets/stocks/most-active/"
}

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
datos_finales = []

try:
    # PROCESAMIENTO DE CADA URL
    for categoria, url in URLS_OBJETIVO.items():
        print(f"\n--- Accediendo a categoría: {categoria.upper()} ---")
        driver.get(url)
        
        # Espera a que la tabla principal aparezca en el DOM
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "tbody"))
        )
        
        # Extraemos las filas de la tabla de resultados
        filas = driver.find_elements(By.CSS_SELECTOR, "tbody tr")
        print(f"Registros encontrados en {categoria}: {len(filas)}")

        for fila in filas:
            try:
                celdas = fila.find_elements(By.TAG_NAME, "td")
                if len(celdas) >= 3:
                    # Capturamos Símbolo (Col 1), Nombre (Col 2) y Precio (Col 3)
                    simbolo = celdas[0].text
                    precio_texto = celdas[2].text
                    
                    item = {
                        "identificador": simbolo,
                        "valor_original": precio_texto,
                        "categoria": categoria,
                        "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                        "grupo": NOMBRE_GRUPO
                    }
                    datos_finales.append(item)
            except:
                continue

    print(f"\nExtracción terminada. Total en memoria: {len(datos_finales)}")

except Exception as e:
    print(f"Error en la arquitectura de scraping: {e}")
finally:
    driver.quit()

# --- ALMACENAMIENTO EN MONGODB ---
try:
    # 'database' debe coincidir con el nombre del servicio en tu docker-compose.yml
    client = MongoClient('database', 27017)
    db = client['proyecto_bigdata']
    coleccion = db['datos_yahoo_finance']
    
    insertados = 0
    for dato in datos_finales:
        try:
            # Limpieza de precio: quitamos comas de miles para convertir a float
            precio_limpio = dato['valor_original'].replace(',', '').strip()
            dato['valor'] = float(precio_limpio)
            
            coleccion.insert_one(dato)
            insertados += 1
        except:
            continue

    print(f"ÉXITO: Se han cargado {insertados} registros en la base de datos.")
except Exception as e:
    print(f"Error de conexión a MongoDB: {e}")